In [6]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
def CSVMerger(rac_file, zeo_file, labels_file, id_col="MOFname"):

    """
    Merge RAC, Zeo++, and QMOF label (bandgap) CSV files into a single dataset.

    This function:
    - Loads three CSV files: RAC descriptors, Zeo++ descriptors, and bandgap labels.
    - Ensures all files contain the same set of MOF identifiers.
    - Drops duplicate MOFs within each file.
    - Merges all three datasets on the MOF identifier.
    - Saves the merged dataset as "merged_rac_zeo_bandgap.csv".
    """

    rac_file, zeo_file, labels_file = map(Path, (rac_file, zeo_file, labels_file))
    out_dir = rac_file.parent

    rac_df = pd.read_csv(rac_file)
    zeo_df = pd.read_csv(zeo_file)
    labels_df = pd.read_csv(labels_file)
    labels_df = labels_df[[id_col, "outputs.pbe.bandgap"]]

    set_rac = set(rac_df[id_col])
    set_zeo = set(zeo_df[id_col])
    set_labels = set(labels_df[id_col])

    # find intersection (common MOFs)
    common_mofs = set_rac & set_zeo & set_labels
    print(f"Number of MOFs common between all three files: {len(common_mofs)}")


    # removing duplciates
    rac  = rac_df.drop_duplicates(subset=[id_col])
    zeo  = zeo_df.drop_duplicates(subset=[id_col])
    qmof = labels_df.drop_duplicates(subset=[id_col])

    # filter to only common MOFs
    rac_c  = rac[rac[id_col].isin(common_mofs)]
    zeo_c  = zeo[zeo[id_col].isin(common_mofs)]
    qmof_c = qmof[qmof[id_col].isin(common_mofs)]
    rac_c = rac_c.rename(columns={"0": "FailedStructures"})

    # merging
    merged_rac_zeo_bandgap = (
        rac_c.merge(zeo_c, on=id_col, how="inner")
            .merge(qmof_c, on=id_col, how="inner")
    )

    #merged_ids = set(merged_rac_zeo_bandgap[id_col])
    return merged_rac_zeo_bandgap

In [4]:
# save the following files in Collab environment before calling: rac_featurization_frame.csv, zeoplus_features.csv, qmof_labels.csv
merged_df = CSVMerger(
    rac_file= Path("/rac_featurization_frame.csv"),
    zeo_file=Path("/zeoplus_features.csv"),
    labels_file=Path("/qmof_labels.csv")
)

Number of MOFs common between all three files: 17879


/tmp/ipython-input-3626220323.py:19: DtypeWarning: Columns (40,41,43,44,45,46,47,49,50,51,52,53,55,56,57,65,66,67,68,69,77,78,79,80,81,89,90,91,92,93) have mixed types. Specify dtype option on import or set low_memory=False.
  labels_df = pd.read_csv(labels_file)


In [7]:
RANDOM_SEED = 10
np.random.seed(RANDOM_SEED)

def DataPreparation(merged_file):

    """
    Takes the merged raw dataset as input, cleans and reduces features,
    splits into train/val/test, saves them to disk, and returns the three DataFrames.
    """
    df = pd.read_csv(merged_file)

    # removing the MOFs with Failed structures/ "Failed to featurize qmof-xx: No linkers were identified.""
    df = df[df["FailedStructures"] != 0].drop(columns=["FailedStructures"])

    target_feature = "outputs.pbe.bandgap"

    cols_to_drop = [
        # columns to drop from the zeoplus file
        "ASA_m^2/cm^3", "ASA_m^2/g", "Largest_included_sphere_along_free_path",
        "NASA_m^2/cm^3", "NASA_m^2/g", "POAV_Volume_fraction",
        "POAV_cm^3/g", "PONAV_Volume_fraction", "PONAV_cm^3/g",
    ]

    df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])
    df = df.dropna().drop_duplicates()

    mof_id = df["MOFname"]
    df = df.drop(columns=["MOFname"], errors="ignore")

    df_train, df_test = train_test_split(df, test_size=0.15, random_state=RANDOM_SEED)


    feature_cols_train = [col for col in df_train.columns if col != target_feature]
    feature_var = df_train[feature_cols_train].var()
    low_var = feature_var[feature_var < 1e-4].index

    df_train = df_train.drop(columns=list(set(low_var)))
    df_test = df_test.drop(columns=list(set(low_var)))

    # target feature correlation
    corr_matrix = df_train.drop(columns=[target_feature]).corr(method= "pearson").abs() # compute correlation matrix for features only
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)) # keep upper triangle to avoid duplicate pairs (.triu)
    threshold = 0.9

    to_drop = set()

    for col in upper.columns:
        for row in upper.index:
            corr_value = upper.loc[row, col]
            if corr_value > threshold:
                corr_row_target = abs(df_train[row].corr(df_train[target_feature]))
                corr_col_target = abs(df_train[col].corr(df_train[target_feature]))
                if corr_row_target >= corr_col_target:
                    to_drop.add(col)
                else:
                    to_drop.add(row)

    df_train = df_train.drop(columns=to_drop)
    df_test = df_test.drop(columns=to_drop)


    if mof_id is not None:
        df_train = pd.concat([df_train, mof_id.loc[df_train.index]], axis=1)
        df_test  = pd.concat([df_test, mof_id.loc[df_test.index]], axis=1)

    return df_train, df_test

In [9]:
# save the following files in Collab environment before calling: /merged_rac_zeo_bandgap.csv (output of merge function)
df_train, df_test = DataPreparation("/merged_rac_zeo_bandgap.csv")

In [19]:
print(df_train.shape,df_test.shape)

(14009, 84) (2473, 84)
